# 02 — Validação da base

**Objetivo:** antes de tirar qualquer conclusão da série, confirmar que os
números batem. Três perguntas, nesta ordem:

1. De quais colunas sai `PercentualdeDevolucao`? Consigo recalcular a conta
   sozinho e bater com a coluna oficial?
2. Qual o erro entre minha conta e a coluna oficial, em quantos meses confere?
3. Dá pra reproduzir um número publicado na imprensa sobre o MED, usando só
   esta base?

Faz parte da [etapa 3](../README.md#progresso) do projeto.

In [1]:
import pandas as pd

URL = "https://raw.githubusercontent.com/ivadivad/med-em-numeros/main/dados/fraude.csv"
df = pd.read_csv(URL)
df["Data"] = pd.to_datetime(df["AnoMes"], format="%Y%m")
df = df.sort_values("Data").reset_index(drop=True)
df.shape

(52, 25)

## 1. Recalculando `PercentualdeDevolucao`

O dicionário de dados (`DICIONARIO.md`) não tinha a fórmula confirmada —
só candidatos. Testei algumas leituras possíveis do nome das colunas e
comparei cada uma contra a coluna oficial `PercentualdeDevolucao`:

- devolvido (valor) / contestado aceito (valor)
- devolvido (quantidade) / contestações aceitas (quantidade)
- só devolução integral / contestado aceito
- devolvido / (devolvido + resíduo não devolvido)

In [2]:
candidatos = {
    "devolvido_valor / contestado_valor":
        (df.ValorPixdevolvidosintegralmente + df.ValorPixdevolvidosparcialmente)
        / df.ValorPixcontestadosaceitos * 100,
    "devolvido_qtde / contestacoes_aceitas_qtde":
        (df.QuantidadedevolvidaintegralmentepormeiodoMED + df.QuantidadedevolvidaparcialmentepormeiodoMED)
        / df.Qtdecontestacoesaceitas * 100,
    "so_integral_valor / contestado_valor":
        df.ValorPixdevolvidosintegralmente / df.ValorPixcontestadosaceitos * 100,
    "devolvido_valor / (devolvido_valor + residual)":
        (df.ValorPixdevolvidosintegralmente + df.ValorPixdevolvidosparcialmente)
        / (df.ValorPixdevolvidosintegralmente + df.ValorPixdevolvidosparcialmente + df.ValorPixresidualnaodevolvido) * 100,
}

for nome, serie in candidatos.items():
    erro = (serie - df.PercentualdeDevolucao).abs()
    print(f"{nome:50s} erro max={erro.max():7.4f} pp   erro medio={erro.mean():7.4f} pp")

devolvido_valor / contestado_valor                 erro max= 0.0049 pp   erro medio= 0.0028 pp
devolvido_qtde / contestacoes_aceitas_qtde         erro max=36.6766 pp   erro medio=23.9466 pp
so_integral_valor / contestado_valor               erro max= 6.8537 pp   erro medio= 2.9273 pp
devolvido_valor / (devolvido_valor + residual)     erro max=23.0760 pp   erro medio=16.5622 pp


**Achei.** `devolvido_valor / contestado_valor` erra no máximo 0,0049 ponto
percentual em qualquer dos 52 meses — é arredondamento, não divergência real.
As outras três leituras erram de 2 a 37 pontos percentuais, longe demais pra
ser coincidência de arredondamento.

$$
\text{PercentualdeDevolucao} = \frac{\text{ValorPixdevolvidosintegralmente} + \text{ValorPixdevolvidosparcialmente}}{\text{ValorPixcontestadosaceitos}} \times 100
$$

Ou seja: é sobre o **valor**, não sobre a quantidade de casos, e o
denominador é o valor **aceito** (já filtrado), não o valor total solicitado
antes do crivo da instituição.

## 2. Medindo o erro

Quantifica o que o teste acima já sugeriu: erro por mês, e em quantos dos
52 meses a reconstrução bate (uso 0,01 ponto percentual como tolerância —
é a casa decimal que a própria coluna usa).

In [3]:
recalculado = (df.ValorPixdevolvidosintegralmente + df.ValorPixdevolvidosparcialmente) / df.ValorPixcontestadosaceitos * 100
erro = (recalculado - df.PercentualdeDevolucao).abs()

print(f"erro maximo:  {erro.max():.4f} pontos percentuais")
print(f"erro medio:   {erro.mean():.4f} pontos percentuais")
print(f"meses dentro de 0,01 pp: {(erro <= 0.01).sum()} de {len(df)}")

erro maximo:  0.0049 pontos percentuais
erro medio:   0.0028 pontos percentuais
meses dentro de 0,01 pp: 52 de 52


**52 de 52 meses batem dentro de 0,01 ponto percentual.** A fórmula está
confirmada, não é só plausível — reproduz a coluna oficial em toda a série,
não em alguns meses escolhidos a dedo.

## 3. Reproduzindo um número da imprensa

Duas reportagens publicadas no mesmo dia (10/09/2024), ambas citando dados do
Banco Central, trazem o mesmo número pro mesmo recorte:

> "entre janeiro e julho de 2024 [...] o valor recuperado por meio do
> mecanismo alcançou R$ 284 milhões, representando aproximadamente 8% do
> total solicitado" — [CNN Brasil](https://www.cnnbrasil.com.br/economia/macroeconomia/mais-r-1-bilhao-em-golpes-e-falhas-no-pix-ja-foram-devolvidos-por-meio-de-mecanismo-especial-do-bc/)

> "a devolução já soma R\$ 952,3 milhões [...] o percentual de devolução [...]
> este ano [2024] é de 8,2%" — [Exame](https://exame.com/economia/pix-banco-central-ja-devolveu-r-11-bi-para-vitimas-de-fraudes-ou-falhas-no-sistema/)

Refazendo a conta da seção 1, só pro recorte jan–jul/2024 citado pela CNN:

In [4]:
periodo = df[(df["AnoMes"] >= 202401) & (df["AnoMes"] <= 202407)]

devolvido = (periodo.ValorPixdevolvidosintegralmente + periodo.ValorPixdevolvidosparcialmente).sum()
contestado = periodo.ValorPixcontestadosaceitos.sum()
percentual = devolvido / contestado * 100

print(f"meses no recorte: {len(periodo)}")
print(f"devolvido:   R$ {devolvido:,.0f}")
print(f"contestado:  R$ {contestado:,.0f}")
print(f"percentual:  {percentual:.2f}%")

meses no recorte: 7
devolvido:   R$ 280,283,578
contestado:  R$ 3,465,083,987
percentual:  8.09%


**R\$ 280 milhões devolvidos / 8,09%** contra os R\$ 284 milhões / ~8% da
reportagem — bateu, com uma diferença pequena o bastante pra ser revisão de
dado entre a data da reportagem e a data desta coleta (o BCB revisa números
publicados; é por isso que o coletor guarda snapshot datado).

**Uma ressalva que vale registrar:** a CNN descreve o denominador como "total
solicitado", mas a fórmula confirmada na seção 1 usa `ValorPixcontestadosaceitos`
— o valor **já aceito** pela instituição, não o total pedido antes do crivo.
Em quantidade de casos, só 42,7% das contestações do período foram aceitas.
Ou seja: a imprensa simplificou a palavra, não a conta — o número bate porque
o cálculo do BCB já é sobre o aceito, e foi isso que a reportagem repassou,
mesmo chamando de "solicitado".

**Base validada em duas frentes agora:** completa (etapa 2) e com os números
batendo — contra a própria coluna oficial e contra uma fonte externa (etapa
3). Dá pra seguir pra análise com confiança na base.